# 🔬 Feature Lab – Quant Researcher
**Workflow**: Load OHLCV → Compute features → Interactive EDA → Feature importance → Correlation Analysis → Save to FeatureStore

---

In [1]:
from qtrader.output.analyst import AnalystSession, RoleContext
from qtrader.input.features.store import FeatureStore
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

session = AnalystSession(role=RoleContext.RESEARCHER)
session.info()

SYMBOL = 'BTC-USD'
TIMEFRAME = '1d'

🔬 Quant Researcher Workflow
  1. load_ohlcv → raw OHLCV
  2. add_rolling_features → compute & store features via FeatureStore
  3. run_alpha_score → forward-return scoring
  4. Use MLflow for experiment tracking (see notebooks/researcher/)
  5. load_features → pull pre-computed features

  📓 Notebooks: notebooks/researcher/01_Feature_Lab.ipynb, ...



## 1. Data Ingestion & Engineering
Load market data and augment with technical factors.

In [2]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME, days=365)
except Exception:
    df = session.sample_ohlcv(symbol='BTC', days=365)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 10, 21, 63])
df = df.drop_nulls()

# Alpha scores (forward returns)
df = session.run_alpha_score(df, forward_periods=[1, 5, 10])
df = df.drop_nulls()

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (658, 24)


timestamp,open,high,low,close,volume,symbol,tf,returns,sma_5,vol_5,sma_10,vol_10,sma_21,vol_21,sma_63,vol_63,avg_gain,avg_loss,rsi_14,fwd_ret_1,fwd_ret_5,fwd_ret_10,alpha_score
"datetime[μs, Asia/Ho_Chi_Minh]",f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2024-05-18 07:00:00 +07,67046.34,67407.79,66624.06,66923.87,3880.58114,"""BTC-USD""","""1d""",-0.001826,65401.382,2273.145391,63606.797,2536.274081,62784.201429,2350.939377,65472.561746,3271.215679,0.012284,0.008602,58.814818,-0.009896,-0.016922,-0.01509,-0.013969
2024-05-19 07:00:00 +07,66923.87,67701.91,65852.02,66261.62,3565.113462,"""BTC-USD""","""1d""",-0.009896,66345.738,713.831457,63925.602,2659.188761,62934.329048,2470.344101,65439.51381,3252.20407,0.012149,0.009309,56.618819,0.078032,0.008939,-0.011014,0.025319
2024-05-20 07:00:00 +07,66260.04,71560.9,66057.03,71432.17,20415.886917,"""BTC-USD""","""1d""",0.078032,67383.234,2372.646805,64990.072,3313.462282,63295.63,3087.838614,65500.134921,3328.081239,0.017723,0.008348,67.979855,-0.01805,0.010823,0.011383,0.001385
2024-05-21 07:00:00 +07,71432.16,71980.0,69146.01,70142.84,16102.061325,"""BTC-USD""","""1d""",-0.01805,68361.366,2280.796709,65922.893,3320.415681,63748.998571,3362.40289,65630.874127,3346.35511,0.017723,0.00868,67.125055,-0.014726,-0.011776,-0.012675,-0.013059
2024-05-22 07:00:00 +07,70142.88,70659.99,68887.52,69109.93,10889.103009,"""BTC-USD""","""1d""",-0.014726,68774.086,2167.225986,66688.584,3046.693213,64265.395714,3310.490099,65650.733333,3363.458919,0.017723,0.008418,67.797832,-0.016922,0.013071,0.003659,-0.000064


## 2. Interactive Correlation Analysis
Examine how features correlate with forward returns across different horizons.

In [3]:
feature_cols = [c for c in df.columns if c.startswith(('sma_', 'vol_', 'rsi_', 'avg_'))]
target_cols  = ['fwd_ret_1', 'fwd_ret_5', 'fwd_ret_10']
df = df.drop_nulls()

corr_matrix = []
for fc in feature_cols:
    row = []
    for tc in target_cols:
        c = np.corrcoef(df[fc], df[tc])[0, 1]
        row.append(c)
    corr_matrix.append(row)

fig = px.imshow(
    corr_matrix, 
    x=target_cols, 
    y=feature_cols, 
    color_continuous_scale='RdBu_r', 
    aspect="auto",
    labels=dict(color="Correlation"),
    title="Feature × Forward Return Correlation Matrix"
)
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117',
    width=800, 
    height=max(400, len(feature_cols)*25)
)
fig.show()

## 3. Dynamic Feature Stability
Check if the predictive power of the top feature is stable over time.

In [4]:
corr_vals = np.array(corr_matrix)
top_idx = np.unravel_index(np.argmax(np.abs(corr_vals)), corr_vals.shape)
top_feat = feature_cols[top_idx[0]]
target_feat = target_cols[top_idx[1]]

print(f"Analyzing Stability for: {top_feat} vs {target_feat}")

win = 90
timestamps = df['timestamp'].to_numpy()[win:]
feat_vals = df[top_feat].to_numpy()
target_vals = df[target_feat].to_numpy()

rolling_corr = [np.corrcoef(feat_vals[i:i+win], target_vals[i:i+win])[0,1] for i in range(len(feat_vals)-win)]

fig_stab = go.Figure()
fig_stab.add_trace(go.Scatter(
    x=timestamps, 
    y=rolling_corr, 
    mode='lines', 
    name=f'Rolling {win}d Corr',
    line=dict(color='#a78bfa', width=2)
))
fig_stab.add_hline(y=0, line_dash="dash", line_color="#64748b")
fig_stab.update_layout(
    title=f"Predictive Stability: {top_feat} vs {target_feat}",
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117',
    xaxis_title="Date",
    yaxis_title="Correlation",
    height=400
)
fig_stab.show()

Analyzing Stability for: sma_63 vs fwd_ret_10


## 4. Feature Distribution & Outliers
Understanding feature behavior across different market conditions.

In [5]:
fig_dist = px.histogram(
    df.to_pandas(), 
    x=top_feat, 
    marginal="box", 
    title=f"Distribution of {top_feat}", 
    color_discrete_sequence=['#34d399']
)
fig_dist.update_layout(
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117'
)
fig_dist.show()

## 5. Persistence
Save engineered features to the unified store.

In [6]:
store = FeatureStore()
save_cols = ['timestamp', 'close', 'returns'] + feature_cols + target_cols
store.save_features(df.select(save_cols), symbol=SYMBOL, timeframe=TIMEFRAME)
print(f"✅ Features persisted to store for {SYMBOL}/{TIMEFRAME}")

✅ Features persisted to store for BTC-USD/1d
